In [1]:
!unzip -o demos_goal.zip

Archive:  demos_goal.zip
   creating: demos_goal/
   creating: demos_goal/goal_demo_7/
  inflating: demos_goal/goal_demo_7/goal_demo_7_0.db3  
  inflating: demos_goal/goal_demo_7/metadata.yaml  
   creating: demos_goal/goal_demo_8/
  inflating: demos_goal/goal_demo_8/metadata.yaml  
  inflating: demos_goal/goal_demo_8/goal_demo_8_0.db3  
   creating: demos_goal/goal_demo_11/
  inflating: demos_goal/goal_demo_11/metadata.yaml  
  inflating: demos_goal/goal_demo_11/goal_demo_11_0.db3  
   creating: demos_goal/goal_demo_23/
  inflating: demos_goal/goal_demo_23/goal_demo_23_0.db3  
  inflating: demos_goal/goal_demo_23/metadata.yaml  
   creating: demos_goal/goal_demo_19/
  inflating: demos_goal/goal_demo_19/metadata.yaml  
  inflating: demos_goal/goal_demo_19/goal_demo_19_0.db3  
   creating: demos_goal/goal_demo_6/
  inflating: demos_goal/goal_demo_6/metadata.yaml  
  inflating: demos_goal/goal_demo_6/goal_demo_6_0.db3  
   creating: demos_goal/goal_demo_18/
  inflating: demos_goal/goal_d

In [2]:
# ==========================================
# 1. 必要ライブラリのインストールとインポート
# ==========================================
!pip install rosbags torch torchvision

import os
import glob
import math
from rosbags.rosbag2 import Reader
from rosbags.typesys import Stores, get_typestore, get_types_from_msg
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

TURTLESIM_POSE_MSG = """
float32 x
float32 y
float32 theta
float32 linear_velocity
float32 angular_velocity
"""

# ==========================================
# 2. ゴールアプローチ用データセット定義
# ==========================================
class GoalApproachDataset(Dataset):
    MAP_SIZE = 11.0
    MAX_LINEAR = 2.0
    MAX_ANGULAR = 2.0
    GOAL_X = 5.5
    GOAL_Y = 5.5

    def __init__(self, demo_dirs):
        self.states = []
        self.actions = []

        self.typestore = get_typestore(Stores.ROS2_HUMBLE)
        self.typestore.register(get_types_from_msg(TURTLESIM_POSE_MSG, 'turtlesim/msg/Pose'))

        for d in demo_dirs:
            if not os.path.exists(d):
                continue
            try:
                pose_msgs = []
                vel_msgs = []
                with Reader(d) as reader:
                    for connection, timestamp, rawdata in reader.messages():
                        if connection.topic == "/turtle1/pose":
                            msg = self.typestore.deserialize_cdr(rawdata, connection.msgtype)
                            pose_msgs.append((timestamp, msg.x, msg.y, msg.theta))
                        elif connection.topic == "/turtle1/cmd_vel":
                            msg = self.typestore.deserialize_cdr(rawdata, connection.msgtype)
                            vel_msgs.append((timestamp, [msg.linear.x, msg.angular.z]))

                if len(pose_msgs) == 0:
                    continue

                pose_msgs.sort(key=lambda x: x[0])

                for t_curr, x, y, theta in pose_msgs:
                    # 固定ゴール（5.5, 5.5）に対する相対座標
                    goal_dx = self.GOAL_X - x
                    goal_dy = self.GOAL_Y - y

                    # 正規化
                    x_norm = x / self.MAP_SIZE
                    y_norm = y / self.MAP_SIZE
                    goal_dx_norm = goal_dx / self.MAP_SIZE
                    goal_dy_norm = goal_dy / self.MAP_SIZE

                    # theta -> sin/cos 変換
                    sin_theta = math.sin(theta)
                    cos_theta = math.cos(theta)

                    # 時間同期フィルタ（0.1秒以内 = 1e8 ns）の最適マッチング
                    valid_vels = [
                        (v_time, v)
                        for v_time, v in vel_msgs
                        if abs(v_time - t_curr) < 1e8
                    ]
                    if len(valid_vels) == 0:
                        continue
                    _, best_vel = min(
                        valid_vels,
                        key=lambda item: abs(item[0] - t_curr)
                    )

                    # 出力（速度）の正規化
                    linear_norm = best_vel[0] / self.MAX_LINEAR
                    angular_norm = best_vel[1] / self.MAX_ANGULAR

                    state_vector = [
                        x_norm,
                        y_norm,
                        sin_theta,
                        cos_theta,
                        goal_dx_norm,
                        goal_dy_norm
                    ]
                    action_vector = [
                        linear_norm,
                        angular_norm
                    ]
                    self.states.append(state_vector)
                    self.actions.append(action_vector)
            except Exception as e:
                print(f"Error reading {d}: {e}")

        if len(self.states) > 0:
            self.states = torch.tensor(self.states, dtype=torch.float32)
            self.actions = torch.tensor(self.actions, dtype=torch.float32)
            print(f"Loaded {len(self.states)} samples successfully.")
        else:
            self.states = torch.zeros((0, 6), dtype=torch.float32)
            self.actions = torch.zeros((0, 2), dtype=torch.float32)
            print("No samples loaded.")

    def __len__(self):
        return len(self.states)

    def __getitem__(self, idx):
        return self.states[idx], self.actions[idx]

# ==========================================
# 3. ポリシーネットワーク定義
# ==========================================
class TurtlePolicyNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(6, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
            nn.Tanh()
        )
    def forward(self, x):
        return self.fc(x)

# ==========================================
# 4. 学習の実行とモデル保存
# ==========================================
def train_model():
    demo_dirs = glob.glob("demos_goal/goal_demo_*")
    if not demo_dirs:
        print("Demo folders not found.")
        return

    dataset = GoalApproachDataset(demo_dirs)
    if len(dataset) == 0:
        print("Dataset is empty.")
        return

    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

    model = TurtlePolicyNetwork()
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 60
    print("Start training (Goal Approach)...")
    for epoch in range(epochs):
        total_loss = 0.0
        for states, actions in dataloader:
            optimizer.zero_grad()
            outputs = model(states)
            loss = criterion(outputs, actions)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} Loss: {total_loss/len(dataloader):.6f}")

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "input_dim": 6,
            "max_linear": 2.0,
            "max_angular": 2.0,
            "map_size": 11.0,
            "goal_x": 5.5,
            "goal_y": 5.5
        },
        "turtle_goal_policy.pth"
    )
    print("Saved: turtle_goal_policy.pth")

if __name__ == "__main__":
    train_model()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.1/151.1 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.4/493.4 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 16.0 MB/s eta 0:00:00
Loaded 31441 samples successfully.
Start training (Goal Approach)...
Epoch 10/60 Loss: 0.000975
Epoch 20/60 Loss: 0.000698
Epoch 30/60 Loss: 0.000580
Epoch 40/60 Loss: 0.000520
Epoch 50/60 Loss: 0.000476
Epoch 60/60 Loss: 0.000426
Saved: turtle_goal_policy.pth
